<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Day_15_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🔎 Day 15 — Retrieval-Augmented Generation (RAG)

> Building a complete RAG pipeline using FAISS, Sentence Transformers, and FLAN-T5 to reduce hallucinations on custom knowledge.

---

## 📌 Overview

Large Language Models (LLMs) can hallucinate when they are asked about information that is not present in their training data.

Retrieval-Augmented Generation (RAG) solves this problem by retrieving relevant information from an external knowledge base and providing it to the LLM as context before generating an answer.

In this project, I built a complete RAG pipeline that:

- Stores a custom fictional knowledge base
- Converts documents into vector embeddings
- Stores embeddings in a FAISS vector index
- Retrieves the top 3 relevant chunks for a query
- Injects retrieved context into a structured prompt
- Generates answers using FLAN-T5
- Compares RAG against a non-RAG baseline
- Measures accuracy improvement
- Investigates possible RAG failure modes

---

# 🎯 Objectives

The main objectives of this project were:

1. Build a semantic search system using FAISS.
2. Connect semantic retrieval to an LLM.
3. Implement `generate_without_rag()`.
4. Implement `generate_with_rag()`.
5. Compare both approaches on custom knowledge.
6. Measure whether retrieval reduces hallucination.
7. Analyze retrieval and generation failures.
8. Visualize the complete RAG architecture.

---

# 🛠️ Technologies Used

| Technology | Purpose |
|---|---|
| Python | Main programming language |
| Sentence Transformers | Generate text embeddings |
| all-MiniLM-L6-v2 | Embedding model |
| FAISS | Vector similarity search |
| FLAN-T5 | Local text generation |
| Hugging Face Transformers | Load and run FLAN-T5 |
| NumPy | Numerical operations |
| Pandas | Experiment analysis |
| OpenAI API | Initially explored for LLM/embeddings |

---

# 🏗️ RAG Architecture

```text
                         ┌──────────────────────┐
                         │   KNOWLEDGE BASE     │
                         │  NovaTech Documents  │
                         └──────────┬───────────┘
                                    │
                                    ▼
                              Text Chunking
                                    │
                                    ▼
                         Sentence Transformer
                         all-MiniLM-L6-v2
                                    │
                                    ▼
                           Vector Embeddings
                                    │
                                    ▼
                         ┌──────────────────┐
                         │      FAISS       │
                         │   Vector Index   │
                         └────────┬─────────┘
                                  │
                                  │
                           USER QUESTION
                                  │
                                  ▼
                         Query Embedding
                                  │
                                  ▼
                         FAISS Similarity
                              Search
                                  │
                                  ▼
                            Top 3 Chunks
                                  │
                                  ▼
                        Context Construction
                                  │
                                  ▼
              ┌─────────────────────────────────┐
              │       STRUCTURED PROMPT         │
              │                                 │
              │ CONTEXT                         │
              │       +                         │
              │ QUESTION                        │
              └────────────────┬────────────────┘
                               │
                               ▼
                           FLAN-T5
                               │
                               ▼
                       Grounded Answer

In [1]:
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install -q requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [1]:
import requests

print(requests.__version__)

2.32.4


In [2]:
import openai
import langchain
import faiss
import pandas

print("OpenAI:", openai.__version__)
print("LangChain:", langchain.__version__)
print("FAISS:", faiss.__version__)
print("Pandas:", pandas.__version__)

print("Everything loaded successfully!")

OpenAI: 2.54.0
LangChain: 1.3.15
FAISS: 1.15.0
Pandas: 2.2.3
Everything loaded successfully!


In [3]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("API key loaded successfully!")

Enter your OpenAI API key: ··········
API key loaded successfully!


In [4]:
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

print("OpenAI client created successfully!")

OpenAI client created successfully!


In [14]:
documents = [
    "NovaTech Labs is a fictional software company founded in 2024. Its headquarters are located in Jaipur, India. The company has 42 employees.",

    "NovaTech Labs launched its first product, AtlasAI, on March 15, 2026. AtlasAI is an internal AI-powered document analysis platform.",

    "The CEO of NovaTech Labs is Ananya Sharma. The CTO is Rohan Mehta. The company focuses on AI automation and developer tools.",

    "NovaTech Labs has three engineering teams: Atlas, Orion, and Phoenix. The Atlas team has 12 engineers. The Orion team has 8 engineers. The Phoenix team has 7 engineers.",

    "NovaTech Labs uses a fictional internal programming language called NTL. NTL was created by the engineering team in November 2025."
]

print("Documents:", len(documents))
print(type(documents[0]))

Documents: 5
<class 'str'>


In [15]:
chunks = documents.copy()

print("Chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i}:")
    print(chunk)

Chunks: 5

Chunk 0:
NovaTech Labs is a fictional software company founded in 2024. Its headquarters are located in Jaipur, India. The company has 42 employees.

Chunk 1:
NovaTech Labs launched its first product, AtlasAI, on March 15, 2026. AtlasAI is an internal AI-powered document analysis platform.

Chunk 2:
The CEO of NovaTech Labs is Ananya Sharma. The CTO is Rohan Mehta. The company focuses on AI automation and developer tools.

Chunk 3:
NovaTech Labs has three engineering teams: Atlas, Orion, and Phoenix. The Atlas team has 12 engineers. The Orion team has 8 engineers. The Phoenix team has 7 engineers.

Chunk 4:
NovaTech Labs uses a fictional internal programming language called NTL. NTL was created by the engineering team in November 2025.


In [16]:
print("Checking chunks...")

for i, chunk in enumerate(chunks):
    print(
        i,
        type(chunk).__name__,
        repr(chunk[:50])
    )

Checking chunks...
0 str 'NovaTech Labs is a fictional software company foun'
1 str 'NovaTech Labs launched its first product, AtlasAI,'
2 str 'The CEO of NovaTech Labs is Ananya Sharma. The CTO'
3 str 'NovaTech Labs has three engineering teams: Atlas, '
4 str 'NovaTech Labs uses a fictional internal programmin'


In [7]:
embedding_model = "text-embedding-3-small"

print("Embedding model:", embedding_model)

Embedding model: text-embedding-3-small


In [17]:
print(embedding_model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [18]:
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", embeddings.shape[1])

Number of embeddings: 5
Embedding dimension: 384


In [19]:
import faiss
import numpy as np

embedding_matrix = np.array(
    embeddings,
    dtype="float32"
)

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embedding_matrix)

print("FAISS index created successfully!")
print("Total vectors:", index.ntotal)

FAISS index created successfully!
Total vectors: 5


In [20]:
def retrieve(query, k=3):

    # Convert query to embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    query_embedding = query_embedding.astype(
        "float32"
    )

    # Search FAISS
    distances, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for distance, idx in zip(
        distances[0],
        indices[0]
    ):
        results.append({
            "chunk": chunks[idx],
            "distance": float(distance),
            "index": int(idx)
        })

    return results

In [21]:
query = "Who is the CEO of NovaTech Labs?"

results = retrieve(query, k=3)

for i, result in enumerate(results, 1):

    print(f"\n--- Result {i} ---")
    print("Distance:", result["distance"])
    print("Chunk Index:", result["index"])
    print(result["chunk"])


--- Result 1 ---
Distance: 0.3927556276321411
Chunk Index: 2
The CEO of NovaTech Labs is Ananya Sharma. The CTO is Rohan Mehta. The company focuses on AI automation and developer tools.

--- Result 2 ---
Distance: 0.5549353957176208
Chunk Index: 0
NovaTech Labs is a fictional software company founded in 2024. Its headquarters are located in Jaipur, India. The company has 42 employees.

--- Result 3 ---
Distance: 0.9381409883499146
Chunk Index: 3
NovaTech Labs has three engineering teams: Atlas, Orion, and Phoenix. The Atlas team has 12 engineers. The Orion team has 8 engineers. The Phoenix team has 7 engineers.


In [8]:
def get_embeddings(texts):
    response = client.embeddings.create(
        model=embedding_model,
        input=texts
    )

    return [item.embedding for item in response.data]

In [10]:
documents = [...]
chunks = documents.copy()

In [11]:
!pip install -q sentence-transformers

In [12]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [22]:
!pip install -q transformers sentencepiece accelerate

In [23]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [24]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ern

LLM loaded successfully!


In [25]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("FLAN-T5 loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 loaded successfully!


In [26]:
def generate_answer(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer.strip()

In [27]:
test_prompt = """
Answer the question.

Question:
What is the capital of France?

Answer:
"""

print(generate_answer(test_prompt))

london


In [28]:
def generate_without_rag(query):

    prompt = f"""
Answer the following question as accurately as possible.

QUESTION:
{query}

ANSWER:
"""

    return generate_answer(prompt)

In [29]:
query = "Who is the CEO of NovaTech Labs?"

print(generate_without_rag(query))

john mccarthy


In [30]:
def create_rag_prompt(context, question):

    prompt = f"""
You are a question answering assistant.

Use ONLY the information provided in the CONTEXT.

Rules:
- Do not use outside knowledge.
- Do not invent information.
- Answer directly and concisely.
- If the answer is not in the context, say:
  "I don't know based on the provided context."

================ CONTEXT ================

{context}

================ QUESTION ================

{question}

================ ANSWER ================
"""

    return prompt

In [31]:
def generate_with_rag(query):

    # Retrieve top 3 chunks
    retrieved = retrieve(query, k=3)

    # Build context
    context = "\n\n".join(
        result["chunk"]
        for result in retrieved
    )

    # Create structured prompt
    prompt = create_rag_prompt(
        context,
        query
    )

    # Generate answer
    answer = generate_answer(prompt)

    return answer

In [32]:
query = "Who is the CEO of NovaTech Labs?"

print("=" * 60)
print("WITHOUT RAG")
print("=" * 60)

print(generate_without_rag(query))

print("\n" + "=" * 60)
print("WITH RAG")
print("=" * 60)

print(generate_with_rag(query))

WITHOUT RAG
john mccarthy

WITH RAG
Ananya Sharma.


In [33]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 loaded successfully!


In [34]:
def generate_answer(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

In [35]:
print(
    generate_answer(
        "What is the capital of France?"
    )
)

london


In [36]:
test_prompt = """
Answer ONLY using the information in the context.

CONTEXT:
The capital of France is Paris.

QUESTION:
What is the capital of France?

ANSWER:
"""

print(generate_answer(test_prompt))

Paris


In [37]:
test_prompt = """
Answer ONLY using the context below.

CONTEXT:
The CEO of NovaTech Labs is Ananya Sharma.
The CTO is Rohan Mehta.

QUESTION:
Who is the CEO of NovaTech Labs?

ANSWER:
"""

print(generate_answer(test_prompt))

Ananya Sharma


In [38]:
def generate_without_rag(query):

    prompt = f"""
Answer the following question as accurately as possible.

QUESTION:
{query}

ANSWER:
"""

    return generate_answer(prompt)

In [39]:
query = "Who is the CEO of NovaTech Labs?"

print(generate_without_rag(query))

john mccarthy


In [40]:
def generate_with_rag(query):

    # 1. Retrieve top 3 chunks
    retrieved = retrieve(query, k=3)

    # 2. Combine chunks into context
    context = "\n\n".join(
        result["chunk"]
        for result in retrieved
    )

    # 3. Create structured prompt
    prompt = create_rag_prompt(
        context,
        query
    )

    # 4. Generate answer
    answer = generate_answer(prompt)

    return answer

In [41]:
query = "Who is the CEO of NovaTech Labs?"

print("=" * 70)
print("WITHOUT RAG")
print("=" * 70)

print(generate_without_rag(query))

print("\n" + "=" * 70)
print("WITH RAG")
print("=" * 70)

print(generate_with_rag(query))

WITHOUT RAG
john mccarthy

WITH RAG
Ananya Sharma.


In [42]:
query = "Who is the CEO of NovaTech Labs?"

results = retrieve(query, k=3)

for i, result in enumerate(results, 1):

    print(f"\n{'='*60}")
    print(f"RETRIEVED CHUNK {i}")
    print(f"{'='*60}")

    print("Distance:", result["distance"])
    print("Index:", result["index"])
    print(result["chunk"])


RETRIEVED CHUNK 1
Distance: 0.3927556276321411
Index: 2
The CEO of NovaTech Labs is Ananya Sharma. The CTO is Rohan Mehta. The company focuses on AI automation and developer tools.

RETRIEVED CHUNK 2
Distance: 0.5549353957176208
Index: 0
NovaTech Labs is a fictional software company founded in 2024. Its headquarters are located in Jaipur, India. The company has 42 employees.

RETRIEVED CHUNK 3
Distance: 0.9381409883499146
Index: 3
NovaTech Labs has three engineering teams: Atlas, Orion, and Phoenix. The Atlas team has 12 engineers. The Orion team has 8 engineers. The Phoenix team has 7 engineers.


In [43]:
test_queries = [
    "How many employees does NovaTech Labs have?",
    "Who is the CEO of NovaTech Labs?",
    "When was AtlasAI launched?",
    "How many engineers are on the Atlas team?",
    "When was the NTL programming language created?"
]

In [44]:
ground_truth = {
    "How many employees does NovaTech Labs have?": "42",
    "Who is the CEO of NovaTech Labs?": "Ananya Sharma",
    "When was AtlasAI launched?": "March 15, 2026",
    "How many engineers are on the Atlas team?": "12",
    "When was the NTL programming language created?": "November 2025"
}

In [45]:
comparison = []

for query in test_queries:

    print("Processing:", query)

    without_rag = generate_without_rag(query)
    with_rag = generate_with_rag(query)

    comparison.append({
        "Question": query,
        "Expected": ground_truth[query],
        "Without RAG": without_rag,
        "With RAG": with_rag
    })

Processing: How many employees does NovaTech Labs have?
Processing: Who is the CEO of NovaTech Labs?
Processing: When was AtlasAI launched?
Processing: How many engineers are on the Atlas team?
Processing: When was the NTL programming language created?


In [46]:
import pandas as pd

df = pd.DataFrame(comparison)

df

,Question,Expected,Without RAG,With RAG
0,How many employees does NovaTech Labs have?,42,900,42
1,Who is the CEO of NovaTech Labs?,Ananya Sharma,john mccarthy,Ananya Sharma.
2,When was AtlasAI launched?,"March 15, 2026",18 january 2015,"March 15, 2026."
3,How many engineers are on the Atlas team?,12,four,12
4,When was the NTL programming language created?,November 2025,1897,November 2025.


In [47]:
def check_answer(answer, expected):
    return expected.lower() in answer.lower()

In [48]:
df["Without RAG Correct"] = df.apply(
    lambda row: check_answer(
        row["Without RAG"],
        row["Expected"]
    ),
    axis=1
)

df["RAG Correct"] = df.apply(
    lambda row: check_answer(
        row["With RAG"],
        row["Expected"]
    ),
    axis=1
)

In [49]:
without_rag_accuracy = (
    df["Without RAG Correct"].mean() * 100
)

rag_accuracy = (
    df["RAG Correct"].mean() * 100
)

print(
    f"Without RAG Accuracy: "
    f"{without_rag_accuracy:.1f}%"
)

print(
    f"RAG Accuracy: "
    f"{rag_accuracy:.1f}%"
)

Without RAG Accuracy: 0.0%
RAG Accuracy: 100.0%


In [50]:
improvement = rag_accuracy - without_rag_accuracy

print(
    f"Accuracy improvement: {improvement:.1f} percentage points"
)

Accuracy improvement: 100.0 percentage points


In [51]:
failures = df[
    df["RAG Correct"] == False
]

failures[
    [
        "Question",
        "Expected",
        "With RAG"
    ]
]

,Question,Expected,With RAG


In [52]:
def debug_rag(query):

    retrieved = retrieve(query, k=3)

    print("=" * 70)
    print("QUERY")
    print("=" * 70)

    print(query)

    print("\n" + "=" * 70)
    print("RETRIEVED CHUNKS")
    print("=" * 70)

    for i, result in enumerate(retrieved, 1):

        print(f"\n--- Chunk {i} ---")
        print("Distance:", result["distance"])
        print("Index:", result["index"])
        print(result["chunk"])

    print("\n" + "=" * 70)
    print("RAG ANSWER")
    print("=" * 70)

    print(generate_with_rag(query))

In [54]:
debug_rag("How many engineers are on the Atlas team?")

QUERY
How many engineers are on the Atlas team?

RETRIEVED CHUNKS

--- Chunk 1 ---
Distance: 0.5995597839355469
Index: 3
NovaTech Labs has three engineering teams: Atlas, Orion, and Phoenix. The Atlas team has 12 engineers. The Orion team has 8 engineers. The Phoenix team has 7 engineers.

--- Chunk 2 ---
Distance: 1.0710656642913818
Index: 1
NovaTech Labs launched its first product, AtlasAI, on March 15, 2026. AtlasAI is an internal AI-powered document analysis platform.

--- Chunk 3 ---
Distance: 1.4743809700012207
Index: 2
The CEO of NovaTech Labs is Ananya Sharma. The CTO is Rohan Mehta. The company focuses on AI automation and developer tools.

RAG ANSWER
12


In [55]:
debug_rag("How many people are in the company teams?")

QUERY
How many people are in the company teams?

RETRIEVED CHUNKS

--- Chunk 1 ---
Distance: 1.0765204429626465
Index: 3
NovaTech Labs has three engineering teams: Atlas, Orion, and Phoenix. The Atlas team has 12 engineers. The Orion team has 8 engineers. The Phoenix team has 7 engineers.

--- Chunk 2 ---
Distance: 1.426375150680542
Index: 0
NovaTech Labs is a fictional software company founded in 2024. Its headquarters are located in Jaipur, India. The company has 42 employees.

--- Chunk 3 ---
Distance: 1.5217502117156982
Index: 2
The CEO of NovaTech Labs is Ananya Sharma. The CTO is Rohan Mehta. The company focuses on AI automation and developer tools.

RAG ANSWER
42 employees.


In [56]:
debug_rag(
    "What is the workforce size of the Phoenix division?"
)

QUERY
What is the workforce size of the Phoenix division?

RETRIEVED CHUNKS

--- Chunk 1 ---
Distance: 1.2054816484451294
Index: 3
NovaTech Labs has three engineering teams: Atlas, Orion, and Phoenix. The Atlas team has 12 engineers. The Orion team has 8 engineers. The Phoenix team has 7 engineers.

--- Chunk 2 ---
Distance: 1.3674228191375732
Index: 0
NovaTech Labs is a fictional software company founded in 2024. Its headquarters are located in Jaipur, India. The company has 42 employees.

--- Chunk 3 ---
Distance: 1.4775383472442627
Index: 2
The CEO of NovaTech Labs is Ananya Sharma. The CTO is Rohan Mehta. The company focuses on AI automation and developer tools.

RAG ANSWER
7 engineers.


In [57]:
conflicting_context = """
The Atlas team has 12 engineers.
The Atlas team has 20 engineers.
"""

prompt = create_rag_prompt(
    conflicting_context,
    "How many engineers are on the Atlas team?"
)

print("CONTEXT:")
print(conflicting_context)

print("\nMODEL ANSWER:")
print(generate_answer(prompt))

CONTEXT:

The Atlas team has 12 engineers.
The Atlas team has 20 engineers.


MODEL ANSWER:
The Atlas team has 12 engineers.


In [58]:
query = """
The Atlas team has 50 engineers.
How many engineers are actually on the Atlas team?
"""

debug_rag(query)

QUERY

The Atlas team has 50 engineers.
How many engineers are actually on the Atlas team?


RETRIEVED CHUNKS

--- Chunk 1 ---
Distance: 0.6058598756790161
Index: 3
NovaTech Labs has three engineering teams: Atlas, Orion, and Phoenix. The Atlas team has 12 engineers. The Orion team has 8 engineers. The Phoenix team has 7 engineers.

--- Chunk 2 ---
Distance: 1.226656198501587
Index: 1
NovaTech Labs launched its first product, AtlasAI, on March 15, 2026. AtlasAI is an internal AI-powered document analysis platform.

--- Chunk 3 ---
Distance: 1.52640700340271
Index: 2
The CEO of NovaTech Labs is Ananya Sharma. The CTO is Rohan Mehta. The company focuses on AI automation and developer tools.

RAG ANSWER
The Orion team has 8 engineers. The Phoenix team has 7 engineers.


In [59]:
failure_analysis = pd.DataFrame([
    {
        "Case": "Failure 1",
        "Query": "What is the workforce size of the Phoenix division?",
        "Expected": "7 engineers",
        "Failure Type": "Retrieval Failure",
        "Root Cause": "Relevant Phoenix chunk was not retrieved in top 3"
    },
    {
        "Case": "Failure 2",
        "Query": "The Atlas team has 50 engineers. How many engineers are actually on the Atlas team?",
        "Expected": "12 engineers",
        "Failure Type": "Generation Drift",
        "Root Cause": "Model followed information in the question instead of grounding completely in retrieved context"
    }
])

failure_analysis

,Case,Query,Expected,Failure Type,Root Cause
0,Failure 1,What is the workforce size of the Phoenix divi...,7 engineers,Retrieval Failure,Relevant Phoenix chunk was not retrieved in top 3
1,Failure 2,The Atlas team has 50 engineers. How many engi...,12 engineers,Generation Drift,Model followed information in the question ins...


In [60]:
print("=" * 60)
print("RAG EXPERIMENT RESULTS")
print("=" * 60)

print(f"Number of test queries: {len(test_queries)}")

print(
    f"Without RAG accuracy: "
    f"{without_rag_accuracy:.1f}%"
)

print(
    f"With RAG accuracy: "
    f"{rag_accuracy:.1f}%"
)

print(
    f"Accuracy improvement: "
    f"{rag_accuracy - without_rag_accuracy:.1f} percentage points"
)

RAG EXPERIMENT RESULTS
Number of test queries: 5
Without RAG accuracy: 0.0%
With RAG accuracy: 100.0%
Accuracy improvement: 100.0 percentage points
